# Notebook 2. Projection Pipeline (Cohort Component Method)

Here is everything about the main notebook for projections. It does four actions.

1. Fetches the baselines and rates from the SQLite database.
2. Specifies the Cohort Component Method (CCM) functions for mortality, fertility, aging, and migration.
3. Projects each of the 50 states and DC over seven years at five-year intervals from 2028 to 2058.
4. Stores the projected rows in the population table with is_projection = 1.

Make sure to execute 01_setup_and_data.ipynb before this file. This notebook assumes that db/projection.db exists and has been populated.

### CCM in One Paragraph

Cohort Component Model involves the projection of the population by making use of three separate components. Mortality leads to survival, where the number of people in each cohort is reduced due to deaths. Fertility leads to babies entering the first age category. Migration involves the use of rates in determining young ages while working ages are determined by labor demand gaps.

In [1]:
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / 'db' / 'projection.db'

## 1. Load baselines and rates from SQL

We pull every table the CCM needs into pandas DataFrames in memory. Processing in pandas is fast enough at roughly 10 seconds for all 50 states, and the code reads more clearly than doing every step as a SQL query.

In [18]:
BASE_YEAR = 2023
PROJECTION_YEARS = (2028, 2033, 2038, 2043, 2048, 2053, 2058)

conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON')

pop = pd.read_sql(
    'SELECT state_code, year, race, origin, sex, age_group, population '
    'FROM population WHERE year = ? AND is_projection = 0',
    conn, params=(BASE_YEAR,),
)
fertility = pd.read_sql('SELECT * FROM fertility_rates', conn)
mortality = pd.read_sql('SELECT * FROM mortality_rates', conn)
lfpr = pd.read_sql('SELECT * FROM labor_force_participation', conn)
employment = pd.read_sql('SELECT * FROM employment_projections', conn)
migration = pd.read_sql('SELECT * FROM migration_estimates', conn)
age_groups = [r[0] for r in conn.execute('SELECT age_group FROM age_groups ORDER BY sort_order')]

print(f'baseline pop rows: {len(pop)}')
print(f'fertility rows: {len(fertility)}')
print(f'mortality rows: {len(mortality)}')
print(f'LFPR rows: {len(lfpr)}')
print(f'employment rows: {len(employment)}')
print(f'migration rows: {len(migration)}')
print(f'age groups: {len(age_groups)} from {age_groups[0]} to {age_groups[-1]}')

baseline pop rows: 14688
fertility rows: 7344
mortality rows: 14688
LFPR rows: 14688
employment rows: 408
migration rows: 14688
age groups: 18 from 0-4 to 85+


## 2. CCM constants

Standard demographic conventions. These are documented in the report.

Sex ratio at birth is about 1.05 males per female, which means a 51.22 percent male share. Unemployment rate is 4 percent, the natural rate used to gross up labor demand. Average LFPR is about 64.75 percent, used to convert the labor gap into the number of people needed.

In [19]:
MALE_SEX_RATIO_AT_BIRTH = 1.05 / 2.05
FEMALE_SEX_RATIO_AT_BIRTH = 1.0 - MALE_SEX_RATIO_AT_BIRTH
UNEMPLOYMENT_RATE = 0.04
AVG_LFPR = 0.6475
STEP_YEARS = 5
YOUNG_AGE_GROUPS = ('0-4', '5-9', '10-14')

## 3. CCM helper functions

### 3.1 Next age bucket (aging helper)

In [4]:
def next_age_group(age_group, ordered_age_groups):
    idx = ordered_age_groups.index(age_group)
    if idx == len(ordered_age_groups) - 1:
        return age_group
    return ordered_age_groups[idx + 1]

### 3.2 Births

Births equal the female population multiplied by the annual fertility rate and by 5 years, summed across female age groups within each race and origin pair. The total is then split by sex using the sex ratio at birth.

In [5]:
def compute_births(pop_df, fertility_df, state_code):
    mothers = pop_df[(pop_df['state_code'] == state_code) & (pop_df['sex'] == 'f')]
    f_rates = fertility_df[fertility_df['state_code'] == state_code]

    merged = mothers.merge(f_rates, on=['state_code','race','origin','age_group'], how='left')
    merged['fertility_rate'] = merged['fertility_rate'].fillna(0.0)
    merged['births'] = merged['population'] * merged['fertility_rate'] * STEP_YEARS

    grouped = merged.groupby(['race','origin'], as_index=False)['births'].sum()

    rows = []
    for _, r in grouped.iterrows():
        total = r['births']
        rows.append({'state_code':state_code,'race':r['race'],'origin':r['origin'],
                     'sex':'m','age_group':'0-4',
                     'population':round(total * MALE_SEX_RATIO_AT_BIRTH)})
        rows.append({'state_code':state_code,'race':r['race'],'origin':r['origin'],
                     'sex':'f','age_group':'0-4',
                     'population':round(total * FEMALE_SEX_RATIO_AT_BIRTH)})
    return pd.DataFrame(rows)

### 3.3 Zero migration projection (survival, aging, and births)

This is the population we would get after 5 years if no one moved in or out. It is the starting point for the migration layer in Section 3.5.

In [6]:
def project_zero_migration(pop_df, fertility_df, mortality_df, state_code, ordered_age_groups):
    state_pop  = pop_df[pop_df['state_code'] == state_code].copy()
    state_mort = mortality_df[mortality_df['state_code'] == state_code]

    merged = state_pop.merge(state_mort, on=['state_code','race','origin','sex','age_group'], how='left')
    merged['mortality_rate'] = merged['mortality_rate'].fillna(0.0)
    merged['survived'] = (merged['population'] * (1.0 - merged['mortality_rate'])).round().astype(int)

    merged['new_age_group'] = merged['age_group'].apply(lambda ag: next_age_group(ag, ordered_age_groups))

    aged = (
        merged.groupby(['state_code','race','origin','sex','new_age_group'], as_index=False)['survived'].sum()
        .rename(columns={'new_age_group':'age_group','survived':'population'})
    )
    aged = aged[aged['age_group'] != '0-4']

    births = compute_births(state_pop, fertility_df, state_code)
    return pd.concat([aged, births], ignore_index=True)

### 3.4 Labor supply, demand, gap, and migrants

In [ ]:
def compute_labor_supply(pop_df, lfpr_df, state_code):
    sp = pop_df[pop_df['state_code'] == state_code]
    sl = lfpr_df[lfpr_df['state_code'] == state_code]
    merged = sp.merge(sl, on=['state_code','race','origin','sex','age_group'], how='left')
    merged['lfpr'] = merged['lfpr'].fillna(0.0)
    return int(round((merged['population'] * merged['lfpr']).sum()))

def compute_labor_gap(labor_supply, total_employment, unemployment_rate=UNEMPLOYMENT_RATE):
    labor_demand = total_employment / (1.0 - unemployment_rate)
    return int(round(labor_demand - labor_supply))


def compute_labor_migrants(labor_gap, avg_lfpr=AVG_LFPR):
    return int(round(labor_gap / avg_lfpr))

### 3.5 Migration allocation across demographic cells

Migrants get distributed in proportion to a weight equal to the absolute migration_rate times population. States with higher migration propensity in a cell get more migrants allocated there. The sign of the total labor_migrants preserves the in or out direction per cell.

In [8]:
def allocate_labor_migrants(pop_df, migration_df, labor_migrants, state_code):
    if labor_migrants == 0:
        return pd.DataFrame(columns=['state_code','race','origin','sex','age_group','allocated_migrants'])

    sp = pop_df[pop_df['state_code'] == state_code]
    sm = migration_df[migration_df['state_code'] == state_code]
    merged = sp.merge(sm, on=['state_code','race','origin','sex','age_group'], how='left')
    merged['migration_rate'] = merged['migration_rate'].fillna(0.0)
    merged['weight'] = (merged['migration_rate'].abs() * merged['population']).clip(lower=0)

    total_weight = merged['weight'].sum()
    if total_weight <= 0:
        merged['share'] = 1.0 / len(merged)
    else:
        merged['share'] = merged['weight'] / total_weight

    merged['allocated_migrants'] = (merged['share'] * labor_migrants).round().astype(int)
    return merged[['state_code','race','origin','sex','age_group','allocated_migrants']]

### 3.6 Full one step projection (with migration)

This combines the zero migration base with labor driven migration for working ages and rate based migration for young ages.

In [9]:
def project_one_step(pop_df, fertility_df, mortality_df, lfpr_df, employment_df, migration_df,
                     state_code, target_year, ordered_age_groups):
    zero_mig = project_zero_migration(pop_df, fertility_df, mortality_df, state_code, ordered_age_groups)

    supply = compute_labor_supply(zero_mig, lfpr_df, state_code)
    emp_row = employment_df[(employment_df['state_code'] == state_code) & (employment_df['year'] == target_year)]
    total_employment = int(emp_row['total_employment'].iloc[0]) if len(emp_row) else 0
    gap = compute_labor_gap(supply, total_employment)
    labor_migrants = compute_labor_migrants(gap)

    alloc = allocate_labor_migrants(zero_mig, migration_df, labor_migrants, state_code)

    state_mig = migration_df[migration_df['state_code'] == state_code]
    merged = zero_mig.merge(state_mig, on=['state_code','race','origin','sex','age_group'], how='left')
    merged['migration_rate'] = merged['migration_rate'].fillna(0.0)
    is_young = merged['age_group'].isin(YOUNG_AGE_GROUPS)
    merged['young_migrants'] = 0
    merged.loc[is_young, 'young_migrants'] = (
        merged.loc[is_young, 'population'] * merged.loc[is_young, 'migration_rate']
    ).round().astype(int)

    merged = merged.merge(alloc, on=['state_code','race','origin','sex','age_group'], how='left')
    merged['allocated_migrants'] = merged['allocated_migrants'].fillna(0).astype(int)

    merged['net_migrants'] = merged['young_migrants'].where(is_young, merged['allocated_migrants'])
    merged['new_population'] = (merged['population'] + merged['net_migrants']).clip(lower=0)

    result = merged[['state_code','race','origin','sex','age_group','new_population']].rename(
        columns={'new_population':'population'}
    ).copy()
    result['year'] = target_year
    result['is_projection'] = 1
    return result

## 4. Run projections for all 50 states and DC

The outer loop walks through the states. For each state the inner loop iterates the 7 five year steps. The previous step's population becomes the baseline for the next step.

In [20]:
t0 = time.time()
states = sorted(pop['state_code'].unique())
print(f'Projecting {len(states)} states across years {PROJECTION_YEARS}')

all_steps = []
for i, state in enumerate(states, 1):
    curr = pop[pop['state_code'] == state].copy()
    for target_year in PROJECTION_YEARS:
        step = project_one_step(
            pop_df=curr, fertility_df=fertility, mortality_df=mortality,
            lfpr_df=lfpr, employment_df=employment, migration_df=migration,
            state_code=state, target_year=target_year, ordered_age_groups=age_groups,
        )
        all_steps.append(step)
        curr = step
    if i % 10 == 0 or i == len(states):
        print(f'  [{i:>3}/{len(states)}] {state} done')

projection_df = pd.concat(all_steps, ignore_index=True)
print(f'\nTotal projection rows {len(projection_df)} in {time.time() - t0:.1f}s')

Projecting 51 states across years (2028, 2033, 2038, 2043, 2048, 2053, 2058)
  [ 10/51] FL done
  [ 20/51] MA done
  [ 30/51] NE done
  [ 40/51] RI done
  [ 50/51] WV done
  [ 51/51] WY done

Total projection rows 102816 in 9.3s


## 5. Write projections back to the database

In [21]:
cols = ['state_code','year','race','origin','sex','age_group','population','is_projection']
df_to_write = projection_df[cols].copy()
df_to_write['population']    = df_to_write['population'].astype(int)
df_to_write['is_projection'] = df_to_write['is_projection'].astype(int)

sql = (
    'INSERT OR REPLACE INTO population '
    '(state_code, year, race, origin, sex, age_group, population, is_projection) '
    'VALUES (?,?,?,?,?,?,?,?)'
)
records = list(df_to_write.itertuples(index=False, name=None))
conn.executemany(sql, records)
conn.commit()
print(f'Wrote {len(records)} rows to population')

Wrote 102816 rows to population


## 6. Summary checks

### 6.1 National totals by year

In [12]:
pd.read_sql("""
    SELECT year, is_projection,
           SUM(total_population)  AS pop,
           SUM(total_labor_force) AS lf
    FROM v_state_year_totals
    GROUP BY year, is_projection
    ORDER BY year
""", conn)

,year,is_projection,pop,lf
0,2018,0,330439019,164835430
1,2023,0,338263983,168756680
2,2028,1,344502920,169506961
3,2033,1,358318870,176122337
4,2038,1,371890234,182964898
5,2043,1,385079445,190114383
6,2048,1,398453082,197638981
7,2053,1,412683523,205548833
8,2058,1,428253588,213849422


### 6.2 Growth vs decline for illustrative states

Sun Belt states grow. West Virginia shrinks. This matches real demographic trends.

In [13]:
pd.read_sql("""
    SELECT state_name, year, total_population
    FROM v_state_year_totals
    WHERE state_code IN ('TX','FL','CA','NY','WV')
      AND year IN (2023, 2058)
    ORDER BY state_name, year
""", conn)

,state_name,year,total_population
0,California,2023,39354846
1,California,2058,48760212
2,Florida,2023,22836838
3,Florida,2058,32892716
4,New York,2023,19766919
5,New York,2058,23918874
6,Texas,2023,30808340
7,Texas,2058,47428518
8,West Virginia,2023,1787766
9,West Virginia,2058,1470001


In [14]:
conn.close()

Done. The population table now has both observed and projected rows. Next open notebook 3 for the plots.